In [ ]:
from pathlib import Path
import os

# Keep relative input and output paths anchored at the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
if not (PROJECT_ROOT / "utils.py").is_file():
    raise RuntimeError(f"Open this notebook from the project root or experiments/: {Path.cwd()}")
os.chdir(PROJECT_ROOT)

from ObjaverseDataset import ObjaverseDataset
from torch.utils.data import DataLoader
from pathlib import Path
from utils import *

Add imports

In [ ]:
from ObjaverseDataset import ObjaverseDataset
from torch.utils.data import DataLoader
from pathlib import Path
from utils import *

Define dataset

In [ ]:
objaverse_dataset = ObjaverseDataset(category="duck", limit=1)
objaverse_dataloader = DataLoader(objaverse_dataset, batch_size=1, shuffle=False, collate_fn=objaverse_dataset.collate_fn)

load the object

In [ ]:
current_object = objaverse_dataset[0] # obj_path, vertices, faces
mesh = current_object["mesh"]
mesh.show()

render object in 6 views



In [ ]:
uid_value = current_object["uid"]
uid = uid_value[0] if isinstance(uid_value, (list, tuple)) else uid_value
print(f"=== Processing UID: {uid} ===")
mesh = current_object["mesh"][0] if isinstance(current_object["mesh"], list) else current_object["mesh"]
output_dir = f"rendered_views/{uid}"
rendered_info = render_n_views(mesh, n_views=6, output_dir=output_dir)
show_rendered_grid(output_dir)

Get masks for every image

In [ ]:
valid_masks_set = []
res1, path1 = rendered_info[0]
print(path1)
xmin, ymin, xmax, ymax = 350,750,650,950
mask = segment_image(path1, xmin, ymin, xmax, ymax)
visualize_segmentation(
    image_path=path1, mask=mask,
    x1=xmin, y1=ymin, x2=xmax, y2=ymax
)
valid_masks_set.append((mask, res1))

res2, path2 = rendered_info[1]
print(path2)
xmin, ymin, xmax, ymax = 650,450,850,650
mask = segment_image(path2, xmin, ymin, xmax, ymax)
visualize_segmentation(
    image_path=path2, mask=mask,
    x1=xmin, y1=ymin, x2=xmax, y2=ymax
)
valid_masks_set.append((mask, res2))
#skip rendered_info[4] since it does not contain the beack
res4, path4 = rendered_info[3]
print(path4)
xmin, ymin, xmax, ymax = 180,280,450,470
mask = segment_image(path4, xmin, ymin, xmax, ymax)
visualize_segmentation(
    image_path=path4, mask=mask,
    x1=xmin, y1=ymin, x2=xmax, y2=ymax
)
valid_masks_set.append((mask, res4))

res5, path5 = rendered_info[4]
print(path4)
xmin, ymin, xmax, ymax = 710,380,910,570
mask = segment_image(path5, xmin, ymin, xmax, ymax)
visualize_segmentation(
    image_path=path5, mask=mask,
    x1=xmin, y1=ymin, x2=xmax, y2=ymax
)
valid_masks_set.append((mask, res5))

res6, path6 = rendered_info[5]
print(path4)
xmin, ymin, xmax, ymax = 390,80,650,270
mask = segment_image(path6, xmin, ymin, xmax, ymax)
visualize_segmentation(
    image_path=path6, mask=mask,
    x1=xmin, y1=ymin, x2=xmax, y2=ymax
)
valid_masks_set.append((mask, res6))

Generate 3d mask from mask views

In [ ]:
dense_mesh = mesh.subdivide()
colored_mesh, mask_submesh = project_masks_to_3d(dense_mesh, valid_masks_set)
display(mask_submesh.show())
display(colored_mesh.show())

Remove the masked part of 3d object

In [ ]:
# Find selected faces (green)
face_mask = np.all(colored_mesh.visual.face_colors[:, :3] == [0, 255, 0], axis=1)


# Expand the mask by neighboring triangles
for _ in range(2):
    neighbors = dense_mesh.face_adjacency
    expanded = face_mask.copy()
    expanded[neighbors[face_mask[neighbors[:, 0]], 1]] = True
    expanded[neighbors[face_mask[neighbors[:, 1]], 0]] = True
    face_mask = expanded

# Delete selected faces
edited_mesh = dense_mesh.submesh([~face_mask], append=True)
# Visualize result
edited_mesh.show()

**Experimental**: Remove remaining small details that were not captured by the mask

In [ ]:
from scipy.spatial import cKDTree
# Geometry of the removed region
removed_mesh = dense_mesh.submesh([face_mask], append=True)

# Split remaining mesh into connected components
components = edited_mesh.split(only_watertight=False)
# Spatial index of removed geometry
tree = cKDTree(removed_mesh.vertices)

# Thresholds relative to the original mesh size
#TODO: ВОТ ТУТ ПОИГРАЙСЯ С ПАРАМЕТРАМИ, ХЗ
extent = dense_mesh.extents.max()
max_size = extent * 0.1
max_distance = extent * 0.0001

clean_components = []

for component in components:
    # Check component size
    is_small = component.extents.max() < max_size

    # Check proximity to deleted region
    distance, _ = tree.query(component.vertices)
    is_near = distance.min() < max_distance

    if is_small and is_near:
        print(f"Removed fragment: {len(component.faces)} faces")
    else:
        clean_components.append(component)

# Rebuild mesh
edited_mesh = trimesh.util.concatenate(clean_components)
edited_mesh.show()


Fill the holes of edited mesh

In [ ]:
# Identify edges between removed and preserved faces in the ORIGINAL dense mesh.
adj = dense_mesh.face_adjacency
cut = face_mask[adj[:, 0]] != face_mask[adj[:, 1]]
cut_edges = dense_mesh.face_adjacency_edges[cut]
cut_midpoints = dense_mesh.vertices[cut_edges].mean(axis=1)

# Find open edges on the CLEANED edited mesh (vertex indices may have changed).
edges, counts = np.unique(edited_mesh.edges_sorted, axis=0, return_counts=True)
open_edges = edges[counts == 1]
open_midpoints = edited_mesh.vertices[open_edges].mean(axis=1)

# Keep only open edges that match the deletion boundary geometrically.
from scipy.spatial import cKDTree
if len(cut_midpoints) and len(open_midpoints):
    distances, _ = cKDTree(cut_midpoints).query(open_midpoints)
    boundary_edges = open_edges[distances < max(1e-8, dense_mesh.extents.max() * 1e-6)]
else:
    boundary_edges = np.empty((0, 2), dtype=int)

print(f"Cut edges in original mesh: {len(cut_edges)}")
print(f"Open edges on cleaned mesh: {len(open_edges)}")
print(f"Remaining cut-boundary edges: {len(boundary_edges)}")

# Visualize the detected boundary as line segments over the edited mesh.
if len(boundary_edges):
    boundary_lines = trimesh.load_path(edited_mesh.vertices[boundary_edges])
    trimesh.Scene([edited_mesh, boundary_lines]).show()

In [ ]:

import networkx as nx

G = nx.Graph()
G.add_edges_from(boundary_edges)

loops = [
    G.subgraph(nodes).copy()
    for nodes in nx.connected_components(G)
]

for i, loop in enumerate(loops):
    closed = all(degree == 2 for _, degree in loop.degree())
    print(f"Component {i}: {loop.number_of_edges()} edges, closed={closed}")


In [ ]:

import networkx as nx
import numpy as np

# Get all open boundary edges of the edited mesh
edges, counts = np.unique(
    edited_mesh.edges_sorted,
    axis=0,
    return_counts=True
)
all_boundary_edges = edges[counts == 1]

# Build graph of all boundary edges
G_all = nx.Graph()
G_all.add_edges_from(all_boundary_edges)

# Find boundary components connected to our cut edges
cut_vertices = set(boundary_edges.flatten())

related_components = [
    G_all.subgraph(nodes).copy()
    for nodes in nx.connected_components(G_all)
    if cut_vertices.intersection(nodes)
]

for i, component in enumerate(related_components):
    degrees = [d for _, d in component.degree()]
    print(
        f"Component {i}: "
        f"{component.number_of_edges()} edges, "
        f"closed={all(d == 2 for d in degrees)}, "
        f"endpoints={degrees.count(1)}, "
        f"branches={sum(d > 2 for d in degrees)}"
    )


In [ ]:
import trimesh
import numpy as np

colors = [
    [255, 0, 0, 255],    # Red
    [0, 255, 0, 255],    # Green
    [0, 0, 255, 255]     # Blue
]

scene = trimesh.Scene()
scene.add_geometry(edited_mesh)

for i, loop in enumerate(related_components):
    edges = np.array(list(loop.edges()), dtype=int)

    # Convert edge vertex indices into 3D line segments
    segments = edited_mesh.vertices[edges]

    path = trimesh.load_path(segments)
    path.colors = np.tile(
        colors[i % len(colors)],
        (len(path.entities), 1)
    )

    scene.add_geometry(path)

scene.show()